# 00 - Setup and Validation

This notebook is the shared prerequisite check for **all** demos in the AI
Governance lab series. Run it first, top to bottom, before opening any of the
`demoN-*.ipynb` notebooks.

It will:

1. Confirm you are logged in via `az login` and show the active subscription.
2. Let you choose/confirm the Azure subscription to use.
3. Prompt for (and persist to `.env`) the resource group and APIM instance
   name that the rest of the labs will use.
4. Verify the APIM instance is reachable and print its gateway URL and SKU.

Nothing here creates or modifies any Azure resources -- it only reads
configuration and validates connectivity.


In [1]:
import sys
sys.path.append("..")

from shared import auth, config, display


## 1. Confirm Azure CLI login

In [2]:
import subprocess

try:
    result = subprocess.run(
        ["az", "account", "show", "-o", "json"],
        capture_output=True, text=True, check=True, timeout=30, shell=True
    )
    display.banner("az login is active.", kind="success")
    print(result.stdout)
except Exception as exc:
    display.banner(
        "Could not confirm az login. Run `az login` in a terminal, then re-run this cell.",
        kind="error",
    )
    raise


## 2. Load or collect configuration

In [ ]:
cfg = config.load_config(interactive=True)
config.validate_config(cfg)
display.header("Current configuration (secrets masked)")
_ = display.show_table([cfg.as_display_dict()])


## 3. Validate the APIM instance is reachable

In [ ]:
from shared import apim

service = apim.get_service(cfg.subscription_id, cfg.resource_group, cfg.apim_name)
gateway_url = service["properties"]["gatewayUrl"]
sku = service.get("sku", {}).get("name", "unknown")

display.banner(f"APIM instance '{cfg.apim_name}' is reachable.", kind="success")
display.show_table([
    {
        "name": service.get("name"),
        "location": service.get("location"),
        "sku": sku,
        "gatewayUrl": gateway_url,
        "provisioningState": service.get("properties", {}).get("provisioningState"),
    }
])

supported_llm_token_limit_skus = {
    "Developer", "Basic", "BasicV2", "Standard", "StandardV2", "Premium", "PremiumV2"
}
if sku not in supported_llm_token_limit_skus:
    display.banner(
        "Note: Demo 1 uses llm-token-limit, which is supported on Developer, Basic, "
        "BasicV2, Standard, StandardV2, Premium, and PremiumV2. It is not available "
        "on Consumption. Verify your SKU before running Demo 1.",
        kind="warning",
    )

supported_pool_skus = {"BasicV2", "StandardV2", "PremiumV2", "Standard", "Premium"}
if sku not in supported_pool_skus:
    display.banner(
        "Note: Demo 4 uses backend pools and circuit breakers, which require "
        "BasicV2, StandardV2, PremiumV2, classic Standard, or classic Premium. "
        "They are not available on Consumption, Developer, or classic Basic.",
        kind="warning",
    )


## 4. Content Safety resource check (required for Demo 3)


In [ ]:
from shared import config as _config

display.header("Content Safety resource (Demo 3 prerequisite)")

if cfg.content_safety_endpoint:
    try:
        _config.validate_content_safety_config(cfg)
        display.banner(
            f"CONTENT_SAFETY_ENDPOINT is set and well-formed: {cfg.content_safety_endpoint}",
            kind="success",
        )
    except ValueError as exc:
        display.banner(str(exc), kind="error")
    if cfg.content_safety_key:
        display.banner("CONTENT_SAFETY_KEY is set; the managed identity role below is not required.", kind="info")
    else:
        display.banner(
            "No CONTENT_SAFETY_KEY set -- grant the APIM system-assigned managed identity the "
            "'Cognitive Services User' role on the Content Safety resource before running Demo 3 "
            "(Content Safety resource -> Access control (IAM) -> Add role assignment).",
            kind="warning",
        )
else:
    display.banner(
        "CONTENT_SAFETY_ENDPOINT is not set yet. This is only required for Demo 3 -- Demos 1 and 2 "
        "do not need it. Demo 3 will prompt for it (and persist it to .env) when you open "
        "demo3-content-safety.ipynb.",
        kind="warning",
    )


## Next steps

Configuration has been captured and persisted to `.env`. You can now open:

- `demo1-token-limits.ipynb` -- Token limits & quota enforcement (complete)
- `demo2-token-metrics.ipynb` -- Token metering & chargeback dimensions (complete)
- `demo3-content-safety.ipynb` -- Content safety: inspect both directions (complete)
- `demo4-resilient-pool.ipynb` -- Resilient backend pools: priority/weight routing, circuit breakers, spillover (complete)
